
# Stock Sentiment Analysis with Agentics + DJIA News Dataset

This notebook demonstrates a **stock sentiment analysis** workflow using the classic Kaggle dataset:
- `Combined_News_DJIA.csv` (daily top 25 news headlines and a daily label indicating whether the **DJIA** went **Up (1)** or **Down (0)** the **next trading day**),
- `RedditNews.csv` (headline/time feed),
- `upload_DJIA_table.csv` (DJIA close data).

We show two paths:
1. **Baseline ML**: TF–IDF + Logistic Regression to predict daily DJIA up/down from headlines.
2. **Agentic LLM** (IBM **Agentics**, optional): Use an LLM to produce **structured sentiment** for each headline, aggregate per day, and evaluate against the daily DJIA label.

> ⚠️ The Agentics section requires configuring an LLM backend (e.g., **OpenAI** API key or a **local Ollama** endpoint). If you skip it, the baseline ML still works.


## 0. Setup

In [1]:

# If you plan to use Agentics + OpenAI, uncomment the install and set OPENAI_API_KEY.
# If you plan to use a local LLM (Ollama), set OLLAMA_HOST and MODEL_NAME.

# !pip install -q agentics pydantic scikit-learn pandas numpy tqdm matplotlib requests

import os, sys, pandas as pd, numpy as np, re, json, math, datetime as dt
from pathlib import Path



## 1. Load & Inspect Data

In [2]:

combined_path = "/Users/boxuanli/Documents/GitHub/Agentics/docs/data/Combined_News_DJIA.csv" 
reddit_path = "/Users/boxuanli/Documents/GitHub/Agentics/docs/data/RedditNews.csv"
djia_path ="/Users/boxuanli/Documents/GitHub/Agentics/docs/data/upload_DJIA_table.csv"

df = pd.read_csv(combined_path)
df['Date'] = pd.to_datetime(df['Date'])
print(df.shape)
df.head(3)


(1989, 27)


,Date,Label,Top1,Top2,Top3,Top4,Top5,Top6,Top7,Top8,...,Top16,Top17,Top18,Top19,Top20,Top21,Top22,Top23,Top24,Top25
0,2008-08-08,0,"b""Georgia 'downs two Russian warplanes' as cou...",b'BREAKING: Musharraf to be impeached.',b'Russia Today: Columns of troops roll into So...,b'Russian tanks are moving towards the capital...,"b""Afghan children raped with 'impunity,' U.N. ...",b'150 Russian tanks have entered South Ossetia...,"b""Breaking: Georgia invades South Ossetia, Rus...","b""The 'enemy combatent' trials are nothing but...",...,b'Georgia Invades South Ossetia - if Russia ge...,b'Al-Qaeda Faces Islamist Backlash',"b'Condoleezza Rice: ""The US would not act to p...",b'This is a busy day: The European Union has ...,"b""Georgia will withdraw 1,000 soldiers from Ir...",b'Why the Pentagon Thinks Attacking Iran is a ...,b'Caucasus in crisis: Georgia invades South Os...,b'Indian shoe manufactory - And again in a se...,b'Visitors Suffering from Mental Illnesses Ban...,"b""No Help for Mexico's Kidnapping Surge"""
1,2008-08-11,1,b'Why wont America and Nato help us? If they w...,b'Bush puts foot down on Georgian conflict',"b""Jewish Georgian minister: Thanks to Israeli ...",b'Georgian army flees in disarray as Russians ...,"b""Olympic opening ceremony fireworks 'faked'""",b'What were the Mossad with fraudulent New Zea...,b'Russia angered by Israeli military sale to G...,b'An American citizen living in S.Ossetia blam...,...,b'Israel and the US behind the Georgian aggres...,"b'""Do not believe TV, neither Russian nor Geor...",b'Riots are still going on in Montreal (Canada...,b'China to overtake US as largest manufacturer',b'War in South Ossetia [PICS]',b'Israeli Physicians Group Condemns State Tort...,b' Russia has just beaten the United States ov...,b'Perhaps *the* question about the Georgia - R...,b'Russia is so much better at war',"b""So this is what it's come to: trading sex fo..."
2,2008-08-12,0,b'Remember that adorable 9-year-old who sang a...,"b""Russia 'ends Georgia operation'""","b'""If we had no sexual harassment we would hav...","b""Al-Qa'eda is losing support in Iraq because ...",b'Ceasefire in Georgia: Putin Outmaneuvers the...,b'Why Microsoft and Intel tried to kill the XO...,b'Stratfor: The Russo-Georgian War and the Bal...,"b""I'm Trying to Get a Sense of This Whole Geor...",...,b'U.S. troops still in Georgia (did you know t...,b'Why Russias response to Georgia was right',"b'Gorbachev accuses U.S. of making a ""serious ...","b'Russia, Georgia, and NATO: Cold War Two'",b'Remember that adorable 62-year-old who led y...,b'War in Georgia: The Israeli connection',b'All signs point to the US encouraging Georgi...,b'Christopher King argues that the US and NATO...,b'America: The New Mexico?',"b""BBC NEWS | Asia-Pacific | Extinction 'by man..."


## 2. Preprocess Headlines

In [3]:

# Combine Top1..Top25 into a long list per day
headline_cols = [c for c in df.columns if c.startswith("Top")]
daily = df[['Date','Label'] + headline_cols].copy()

# basic cleaning
def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = s.strip()
    s = re.sub(r"\s+", " ", s)
    return s

for c in headline_cols:
    daily[c] = daily[c].astype(str).map(clean_text)

# Make a long dataframe: one row per (Date, headline)
long_rows = []
for _, row in daily.iterrows():
    date = row['Date']
    label = int(row['Label'])
    headlines = [row[c] for c in headline_cols]
    for h in headlines:
        if h and h != "nan":
            long_rows.append({"Date": date, "Label": label, "headline": h})
long_df = pd.DataFrame(long_rows)
print("Long headlines:", long_df.shape)
long_df.head(5)


Long headlines: (49718, 3)


,Date,Label,headline
0,2008-08-08,0,"b""Georgia 'downs two Russian warplanes' as cou..."
1,2008-08-08,0,b'BREAKING: Musharraf to be impeached.'
2,2008-08-08,0,b'Russia Today: Columns of troops roll into So...
3,2008-08-08,0,b'Russian tanks are moving towards the capital...
4,2008-08-08,0,"b""Afghan children raped with 'impunity,' U.N. ..."


## 3. Baseline ML: TF–IDF + Logistic Regression (Daily Label)

In [4]:

from sklearn.model_selection import TimeSeriesSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import numpy as np

# Aggregate headlines per day into a single document
agg_df = long_df.groupby(['Date','Label'])['headline'].apply(lambda s: " \n ".join(s)).reset_index()
agg_df = agg_df.sort_values('Date').reset_index(drop=True)

# Train/test split by time
cut = int(len(agg_df)*0.8)
train_df = agg_df.iloc[:cut]
test_df = agg_df.iloc[cut:]

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2))),
    ('clf', LogisticRegression(max_iter=1000))
])

pipe.fit(train_df['headline'], train_df['Label'])
pred = pipe.predict(test_df['headline'])
proba = pipe.predict_proba(test_df['headline'])[:,1]

acc = accuracy_score(test_df['Label'], pred)
f1 = f1_score(test_df['Label'], pred)
try:
    auc = roc_auc_score(test_df['Label'], proba)
except:
    auc = float('nan')

print(f"Accuracy: {acc:.3f} | F1: {f1:.3f} | AUC: {auc:.3f}")
print("\nClassification report:\n", classification_report(test_df['Label'], pred, digits=3))


Accuracy: 0.485 | F1: 0.625 | AUC: 0.510

Classification report:
               precision    recall  f1-score   support

           0      0.415     0.112     0.177       196
           1      0.496     0.847     0.625       202

    accuracy                          0.485       398
   macro avg      0.455     0.479     0.401       398
weighted avg      0.456     0.485     0.404       398



## 4. Agentic LLM Labeling with Agentics (Optional)


This section shows how to use **IBM Agentics** to produce **structured sentiment** for each headline, then
aggregate to a daily sentiment score and evaluate against the DJIA label.

You need:
- Either **OpenAI** credentials (`OPENAI_API_KEY`) or a **local Ollama** server (`OLLAMA_HOST`) with a model name.
- The `agentics` package installed.

We run on a **small sample** to control cost/time; adjust `N_DAYS` as desired.


In [7]:

import os, math, pandas as pd

MODEL_NAME = os.environ.get("AGENTICS_MODEL_NAME", "llama3")


# Define a typed output for agentic sentiment
if 'BaseModel' not in globals():
    from pydantic import BaseModel, Field

class SentimentDoc(BaseModel):
    label: str = Field(description="One of: Positive, Negative, Neutral")
    score: float = Field(description="A numeric polarity between -1 (very negative) and +1 (very positive)")
    rationale: str = Field(description="Short explanation focusing on financial/market impact")

def agentics_label_headlines(headlines):
    """Label a list of headlines using Agentics, return list[SentimentDoc].
    If Agentics is not available, fall back to a rule-based stub as placeholder.
    """
    results = []
    try:
        # Minimal Agentics usage: create an agent with typed output and a financial sentiment instruction.
        agent = AG(
            atype=SentimentDoc,
            llm=LLM(model=os.getenv("GEMINI_MODEL_ID"),
                 temperature=0.7,),
            system="""You are a seasoned financial news sentiment analyst.
Return sentiment strictly for stock market impact. Output must obey the typed schema."""
        )
        # Use the << operator (logical transduction) if available, otherwise call agent directly
        try:
            agent = __import__('asyncio').get_event_loop().run_until_complete(agent << headlines)  # batch
            results = agent.states  # list[SentimentDoc]
        except Exception:
            # Fallback: map one by one synchronously
            for h in headlines:
                a = __import__('asyncio').get_event_loop().run_until_complete(agent << [h])
                results.append(a.states[0])
    except Exception as e:
        print("Agentics call failed, falling back to stub. Error:", e)
        HAVE_AGENTICS = False

   
    return results

# Choose a small sample of days to run agentically
N_DAYS = 50
sample = daily.sort_values('Date').head(N_DAYS)
sample_dates = sample['Date'].unique().tolist()

records = []
for d in sample_dates:
    rows = sample[sample['Date']==d]
    headlines = [rows[c].iloc[0] for c in [c for c in daily.columns if c.startswith('Top')]]
    sents = agentics_label_headlines(headlines)
    # aggregate: mean score, majority label
    mean_score = float(sum(s.score for s in sents) / max(len(sents),1))
    labels = [s.label for s in sents]
    maj = max(set(labels), key=labels.count)
    djia_label = int(rows['Label'].iloc[0])
    records.append({
        "Date": pd.to_datetime(d),
        "agentic_mean_score": mean_score,
        "agentic_majority": maj,
        "djia_label": djia_label
    })

agentic_df = pd.DataFrame(records).sort_values("Date")
agentic_df.head()


Agentics call failed, falling back to stub. Error: name 'AG' is not defined


ValueError: max() iterable argument is empty

In [ ]:
# --- Agentics labeling (robust) ---
import os, math, pandas as pd
from typing import Optional, List
from collections import Counter
from pydantic import BaseModel, Field
from agentics import Agentics as AG
    HAVE_AGENTICS = True
except Exception as e:
    print("Agentics not available:", e)
    HAVE_AGENTICS = False

class SentimentDoc(BaseModel):
    label: str = Field(description="One of: Positive, Negative, Neutral")
    score: float = Field(description="A numeric polarity between -1 and +1")
    rationale: str = Field(description="Short explanation focusing on financial/market impact")

def majority_label(labels: List[str]) -> Optional[str]:
    if not labels:
        return None
    return Counter(labels).most_common(1)[0][0]

def agentics_label_headlines(headlines: List[str]) -> List[SentimentDoc]:
    results: List[SentimentDoc] = []
    try:
        agent = AG(
            atype=SentimentDoc,
            llm=LLM(model=os.getenv("GEMINI_MODEL_ID"),
                temperature=0.7,),
            system=(
                "You are a seasoned financial news sentiment analyst. "
                "Return sentiment strictly for stock market impact. "
                "Output must obey the typed schema."
            )
        )
        # Batch with << if available; otherwise do one-by-one
        import asyncio
        try:
            agent = asyncio.get_event_loop().run_until_complete(agent << headlines)
            results = agent.states  # list[SentimentDoc]
        except Exception:
            for h in headlines:
                a = asyncio.get_event_loop().run_until_complete(agent << [h])
                results.append(a.states[0])
    except Exception as e:
        print("Agentics call failed, falling back to stub. Error:", e)
    return results

# Choose a sample size (adjust as you like)
N_DAYS = 50
sample = daily.sort_values('Date').head(N_DAYS)
sample_dates = sample['Date'].dt.normalize().unique().tolist()

records = []
top_cols = [c for c in daily.columns if c.startswith('Top')]

for d in sample_dates:
    rows = sample[sample['Date'].dt.normalize() == d]

    # Collect and clean headlines for the day
    raw = [rows[c].iloc[0] if len(rows[c]) else "" for c in top_cols]
    headlines = [ (h if isinstance(h, str) else "").strip() for h in raw ]
    headlines = [ h for h in headlines if h and h.lower() != "nan" ]

    # Skip days with no usable headlines
    if not headlines:
        # Optionally log/record a neutral placeholder
        # print(f"Skipping {pd.to_datetime(d).date()} (no headlines)")
        continue

    sents = agentics_label_headlines(headlines)
    if not sents:
        # Shouldn't happen, but guard anyway
        continue

    mean_score = float(sum(s.score for s in sents) / max(len(sents), 1))
    maj = majority_label([s.label for s in sents]) or "Neutral"
    djia_label = int(rows['Label'].iloc[0])

    records.append({
        "Date": pd.to_datetime(d),
        "agentic_mean_score": mean_score,
        "agentic_majority": maj,
        "djia_label": djia_label
    })

agentic_df = pd.DataFrame(records).sort_values("Date").reset_index(drop=True)
agentic_df.head()

Agentics not available: cannot import name 'Agentics' from 'agentics' (/Users/boxuanli/Library/Caches/pypoetry/virtualenvs/agentics-py-TUSS0C_V-py3.12/lib/python3.12/site-packages/agentics/__init__.py)
Agentics call failed, falling back to stub. Error: name 'AG' is not defined
Agentics call failed, falling back to stub. Error: name 'AG' is not defined
Agentics call failed, falling back to stub. Error: name 'AG' is not defined
Agentics call failed, falling back to stub. Error: name 'AG' is not defined
Agentics call failed, falling back to stub. Error: name 'AG' is not defined
Agentics call failed, falling back to stub. Error: name 'AG' is not defined
Agentics call failed, falling back to stub. Error: name 'AG' is not defined
Agentics call failed, falling back to stub. Error: name 'AG' is not defined
Agentics call failed, falling back to stub. Error: name 'AG' is not defined
Agentics call failed, falling back to stub. Error: name 'AG' is not defined
Agentics call failed, falling back to 

KeyError: 'Date'

## 5. Evaluate Agentic Aggregate vs DJIA Daily Label

In [ ]:

from sklearn.metrics import roc_auc_score, accuracy_score

# Simple mapping: positive if mean_score > thresh
thresh = 0.0
pred_label = (agentic_df['agentic_mean_score'] > thresh).astype(int)
acc = accuracy_score(agentic_df['djia_label'], pred_label)
try:
    auc = roc_auc_score(agentic_df['djia_label'], agentic_df['agentic_mean_score'])
except Exception:
    auc = float('nan')

print(f"Agentic agg Accuracy: {acc:.3f} | AUC: {auc:.3f}")
agentic_df.head(10)


## 6. Plot Agentic Mean Score Over Time

In [ ]:

import matplotlib.pyplot as plt

plt.figure()
plt.plot(agentic_df['Date'], agentic_df['agentic_mean_score'], label='Agentic Mean Sentiment')
plt.axhline(0, linestyle='--')
plt.title('Agentic Mean Sentiment (First N Days)')
plt.xlabel('Date'); plt.ylabel('Sentiment Score')
plt.show()


## 7. Save Outputs

In [ ]:

out_path = Path("./agentic_results.csv")
agentic_df.to_csv(out_path, index=False)
print("Saved:", out_path.resolve())
